# HW6: Building an Agent With Tools

Run every cell from the top. **Everything already works.**

**Out:** Week 13, Class 1 · **Due:** Week 14, Class 1 · **100 points** · individual work

Work through the notebook and fill in each YOUR TURN cell. Submit this
`.ipynb` with all cells run and their output visible. There is no test to
pass: you are graded on the code working and on your short written answers.

The reasoning policy is yours to write and it does NOT need a language
model: a rule-based policy earns full marks. The assignment is about the
loop, the tools and the failure handling.

Today you will:

1. Build a ReAct loop that chooses between real tools.
2. Make it fail, and add the guard that catches the failure.
3. Argue, with numbers, whether an agent was the right choice.

There is no test to run and nothing to submit. Each task tells you what
you should see when it is right.

In [ ]:
# Setup.
import re
import pandas as pd
import matplotlib.pyplot as plt

def calculator(expression):
    """Evaluate arithmetic, e.g. '12 * 7'. Returns a number as a string."""
    return str(eval(expression, {"__builtins__": {}}, {}))

def price(symbol):
    """Share price by ticker, e.g. 'ACME'. Returns 'unknown symbol' if absent."""
    return str({"ACME": 42.50, "GLOBEX": 118.20}.get(symbol.upper(), "unknown symbol"))

def headcount(department):
    """People in a department, e.g. 'sales'. Returns 'unknown department'."""
    return str({"sales": 24, "engineering": 61}.get(department.lower(), "unknown department"))

TOOLS = {"calculator": calculator, "price": price, "headcount": headcount}

TASKS = [
    ("What is 318 times 47?", "14946"),
    ("What is the ACME share price?", "42.5"),
    ("How many people work in engineering?", "61"),
    ("What is 1000 divided by 8?", "125"),
    ("What is the GLOBEX share price?", "118.2"),
]
print(f"{len(TOOLS)} tools, {len(TASKS)} tasks")

## Part 1. The loop (40 points)

Thought, Action, Observation, repeat. Write the policy that decides.

In [ ]:
# ================== YOUR TURN 1 ==================
# Write policy(question, history) returning a dict with keys thought,
# action and answer. action is either None or (tool_name, argument).
#
# (25 points)
#
# Expected: each of the five tasks routes to the right tool. A rule-based policy
#           is fine: this question is about the interface, not about cleverness.
# ===============================================
def policy(question, history):
    """Decide the next step. Return {'thought':..., 'action':..., 'answer':...}."""
    if history:
        return {"thought": "I have an observation.", "action": None,
                "answer": history[-1]["observation"]}
    # <-- your routing rules here
    return {"thought": "No tool fits.", "action": None, "answer": "I don't know."}

def run_agent(question, max_steps=4, verbose=False):
    history = []
    for _ in range(max_steps):
        d = policy(question, history)
        if verbose:
            print(f"   Thought: {d['thought']}")
        if d["action"] is None:
            return d["answer"], history
        name, arg = d["action"]
        obs = TOOLS[name](arg) if name in TOOLS else f"no such tool: {name}"
        history.append({"action": name, "arg": arg, "observation": obs})
        if verbose:
            print(f"   Action : {name}({arg!r})\n   Observe: {obs}")
    return None, history

for q, expected in TASKS:
    got, _ = run_agent(q)
    print(f"   {str(got == expected):<6} {q[:38]:<40} got {got}")

In [ ]:
# ================== YOUR TURN 2 ==================
# Report accuracy and the average number of tool calls per task.
#
# (15 points)
#
# Expected: a correct agent scores 1.00 with exactly one tool call per task.
#           More than one call per task means your policy is not recognising the
#           observation it already has.
# ===============================================
correct = calls = 0
for q, expected in TASKS:
    got, history = run_agent(q)
    correct += (got == expected)
    calls += len(history)

print(f"accuracy        : {correct / len(TASKS):.2f}")
print(f"tool calls/task : {calls / len(TASKS):.2f}")

## Part 2. Make it fail, then catch it (35 points)

Every one of these will happen in a real system. Handle them deliberately.

In [ ]:
# ================== YOUR TURN 3 ==================
# Run these three broken inputs and record what your agent does.
#
# (15 points)
#
# Expected: at least one crashes or returns something nonsensical. Record
#           exactly what happened before you fix anything: an unrecorded failure
#           is not a finding.
# ===============================================
BROKEN = [
    "What is the price of Apple Inc?",     # a company name, not a ticker
    "What is 5 divided by 0?",             # the tool will raise
    "How many people work in catering?",   # not in the database
]
for q in BROKEN:
    try:
        got, history = run_agent(q)
        print(f"   ok    {q[:38]:<40} -> {got}")
    except Exception as e:
        print(f"   CRASH {q[:38]:<40} -> {type(e).__name__}: {e}")

In [ ]:
# ================== YOUR TURN 4 ==================
# Add guards so none of the three crashes and each returns a useful
# message. Do not silence the errors: report them.
#
# (20 points)
#
# Expected: all three return a sentence a user could act on, such as 'I could not
#           find a ticker called Apple Inc'. Catching an exception and returning
#           None scores zero: that just moves the crash somewhere else.
# ===============================================
def safe_tool(name, arg):
    """Call a tool, converting failures into a message the agent can use."""
    return TOOLS[name](arg)          # <-- add error handling

for q in BROKEN:
    # <-- run your guarded agent on each
    print(f"   {q[:40]:<42} -> (fill this in)")

## Part 3. Was an agent the right call? (25 points)

The honest answer for many tasks is no. Make the argument with numbers.

In [ ]:
# ================== YOUR TURN 5 ==================
# Write a plain WORKFLOW that answers the same five tasks with a fixed
# if/else, and count its steps against your agent's.
#
# (15 points)
#
# Expected: the workflow gets the same answers with fewer steps, because these
#           five tasks fall into three known categories. That is the expected
#           result, not a failure of your agent.
# ===============================================
def workflow(question):
    """No loop, no policy: one pass of fixed rules."""
    return None          # <-- your code here

correct = 0
for q, expected in TASKS:
    correct += (workflow(q) == expected)
print(f"workflow accuracy: {correct / len(TASKS):.2f}")
print(f"workflow steps/task: 1.00   (agent used the number you measured in Q2)")

In [ ]:
# ================== YOUR TURN 6 ==================
# Describe a task where the agent WOULD be worth it, and say what it is
# about that task that the workflow cannot handle.
#
# (10 points)
#
# Expected: a good answer names a task where the number or order of steps is not
#           known in advance, for example one where the result of one lookup
#           determines which tool is needed next. 'It is more flexible' is not an
#           answer.
# ===============================================
# YOUR ANSWER (4 to 6 sentences):
ANSWER_6 = """
"""
print(ANSWER_6.strip() or "not yet")

## Answers

Try each task before reading.

In [ ]:
# Marking
#   Q1  25   policy routes all five tasks correctly
#   Q2  15   accuracy and calls-per-task reported
#   Q3  15   all three failures observed and recorded honestly
#   Q4  20   all three guarded, with useful messages, errors not swallowed
#   Q5  15   a working workflow and an honest comparison
#   Q6  10   a concrete task an agent earns, with the reason
#
# Where students lose marks:
#   - a policy that never inspects `history`, so the loop runs to max_steps
#     every time and the answer is None
#   - except: pass in Q4, which converts a crash into a silent wrong answer
#   - Q6 answered with "agents are more general", which names no task